In [1]:
import zipfile
import os

zip_file_path = 'Basics of BERT and XLM-RoBERTa - PyTorch - 2.zip'
extraction_path = '/unzipped_data'

# Create the extraction directory if it doesn't exist
os.makedirs(extraction_path, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print(f"'{zip_file_path}' unzipped to '{extraction_path}'")

# List the contents of the unzipped directory to see what's inside
print("Contents of unzipped_data:")
for root, dirs, files in os.walk(extraction_path):
    for name in files:
        print(os.path.join(root, name))
    for name in dirs:
        print(os.path.join(root, name))

'Basics of BERT and XLM-RoBERTa - PyTorch - 2.zip' unzipped to '/unzipped_data'
Contents of unzipped_data:
/unzipped_data\Basics of BERT and XLM-RoBERTa - PyTorch
/unzipped_data\Basics of BERT and XLM-RoBERTa - PyTorch\sample_submission.csv
/unzipped_data\Basics of BERT and XLM-RoBERTa - PyTorch\test.csv.zip
/unzipped_data\Basics of BERT and XLM-RoBERTa - PyTorch\train.csv.zip


Now that the data is extracted, let's implement the **Single-Head Attention** module as requested.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.functional as F

class SingleHeadAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim

        # Linear projections for Query, Key, Value
        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.value_proj = nn.Linear(hidden_dim, hidden_dim)

        # Scale factor for the attention scores
        self.scale = torch.sqrt(torch.tensor(hidden_dim, dtype=torch.float32))

    def forward(self, query, key, value, mask=None):
        # Apply linear projections
        Q = self.query_proj(query)
        K = self.key_proj(key)
        V = self.value_proj(value)

        # Calculate attention scores (dot product of Q and K^T)
        # Q: (batch_size, query_len, hidden_dim)
        # K.transpose(-2, -1): (batch_size, hidden_dim, key_len)
        # energy: (batch_size, query_len, key_len)
        energy = torch.matmul(Q, K.transpose(-2, -1)) / self.scale

        # Apply mask if provided (for preventing attention to padded tokens)
        if mask is not None:
            # Fill masked positions with a very small number, so softmax turns them to 0
            energy = energy.masked_fill(mask == 0, -1e10)

        # Apply softmax to get attention weights
        attention_weights = F.softmax(energy, dim=-1)

        # Log attention weights for inspection (e.g., in a training loop)
        # print("Attention Weights Shape:", attention_weights.shape)
        # print("Sample Attention Weights (first batch, first query):")
        # print(attention_weights[0, 0, :5]) # Print first 5 weights for first query in first batch

        # Multiply attention weights with Value to get the output
        # attention_weights: (batch_size, query_len, key_len)
        # V: (batch_size, value_len, hidden_dim) (where key_len == value_len)
        # x: (batch_size, query_len, hidden_dim)
        x = torch.matmul(attention_weights, V)

        return x, attention_weights

# --- Validate shapes with dummy tensors ---
print("\n--- Single-Head Attention Validation ---")
batch_size = 2
seq_len = 10 # Example sequence length for query
key_value_len = 12 # Example sequence length for key/value (can be different from query_len)
hidden_dim = 64

# Dummy tensors
dummy_query = torch.randn(batch_size, seq_len, hidden_dim)
dummy_key = torch.randn(batch_size, key_value_len, hidden_dim)
dummy_value = torch.randn(batch_size, key_value_len, hidden_dim)

# Dummy mask (e.g., for padding)
dummy_mask = torch.ones(batch_size, seq_len, key_value_len)
# Example: mask out the last 2 tokens of the key/value sequence for the first batch
dummy_mask[0, :, -2:] = 0

single_head_attention = SingleHeadAttention(hidden_dim)
output, weights = single_head_attention(dummy_query, dummy_key, dummy_value, mask=dummy_mask)

print(f"Input Query shape: {dummy_query.shape}")
print(f"Input Key shape: {dummy_key.shape}")
print(f"Input Value shape: {dummy_value.shape}")
print(f"Output shape: {output.shape}") # Should be (batch_size, query_len, hidden_dim)
print(f"Attention Weights shape: {weights.shape}") # Should be (batch_size, query_len, key_value_len)

assert output.shape == (batch_size, seq_len, hidden_dim)
assert weights.shape == (batch_size, seq_len, key_value_len)
print("Single-Head Attention: Shapes validated successfully!")


--- Single-Head Attention Validation ---
Input Query shape: torch.Size([2, 10, 64])
Input Key shape: torch.Size([2, 12, 64])
Input Value shape: torch.Size([2, 12, 64])
Output shape: torch.Size([2, 10, 64])
Attention Weights shape: torch.Size([2, 10, 12])
Single-Head Attention: Shapes validated successfully!


Next, let's implement the **Multi-Head Attention Module**, building upon the single-head attention concept.

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout_rate=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        assert hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"

        # Linear projections for Query, Key, Value for all heads combined
        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.value_proj = nn.Linear(hidden_dim, hidden_dim)

        self.fc_out = nn.Linear(hidden_dim, hidden_dim)

        self.dropout = nn.Dropout(dropout_rate)

        self.scale = torch.sqrt(torch.tensor(self.head_dim, dtype=torch.float32))

    def forward(self, query, key, value, mask=None):
        batch_size = query.shape[0]

        # Project Q, K, V
        # (batch_size, seq_len, hidden_dim) -> (batch_size, seq_len, hidden_dim)
        Q = self.query_proj(query)
        K = self.key_proj(key)
        V = self.value_proj(value)

        # Reshape to (batch_size, seq_len, num_heads, head_dim) and then permute
        # Permute to (batch_size, num_heads, seq_len, head_dim) to enable batch matrix multiplication per head
        Q = Q.view(batch_size, -1, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        K = K.view(batch_size, -1, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        V = V.view(batch_size, -1, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        # Calculate attention scores
        # (batch_size, num_heads, query_len, head_dim) @ (batch_size, num_heads, head_dim, key_len)
        # -> (batch_size, num_heads, query_len, key_len)
        energy = torch.matmul(Q, K.permute(0, 1, 3, 2)) / self.scale

        if mask is not None:
            # Apply mask to attention scores. Mask needs to be broadcastable.
            # (batch_size, 1, 1, key_len) or (batch_size, 1, query_len, key_len)
            energy = energy.masked_fill(mask == 0, -1e10)

        # Apply softmax and dropout
        attention_weights = F.softmax(energy, dim=-1)
        attention_weights = self.dropout(attention_weights)

        # Multiply attention weights with V
        # (batch_size, num_heads, query_len, key_len) @ (batch_size, num_heads, value_len, head_dim)
        # -> (batch_size, num_heads, query_len, head_dim)
        x = torch.matmul(attention_weights, V)

        # Concatenate heads
        # (batch_size, num_heads, query_len, head_dim) -> (batch_size, query_len, num_heads, head_dim)
        # -> (batch_size, query_len, hidden_dim)
        x = x.permute(0, 2, 1, 3).contiguous().view(batch_size, -1, self.hidden_dim)

        # Final linear projection
        x = self.fc_out(x)

        return x, attention_weights

# --- Provide a forward example showing input/output shapes ---
print("\n--- Multi-Head Attention Validation ---")
batch_size = 2
seq_len = 10
hidden_dim = 256 # Must be divisible by num_heads
num_heads = 8
dropout_rate = 0.1

# Dummy input tensors (query, key, value are typically the same for self-attention)
dummy_input = torch.randn(batch_size, seq_len, hidden_dim)

# Dummy mask (e.g., for padding)
dummy_mask = torch.ones(batch_size, 1, 1, seq_len) # (batch_size, 1, 1, key_len) for broadcast
dummy_mask[0, :, :, -2:] = 0 # Mask last 2 tokens for the first batch

multi_head_attention = MultiHeadAttention(hidden_dim, num_heads, dropout_rate)
output, weights = multi_head_attention(dummy_input, dummy_input, dummy_input, mask=dummy_mask)

print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}") # Should be (batch_size, seq_len, hidden_dim)
print(f"Attention Weights shape: {weights.shape}") # Should be (batch_size, num_heads, seq_len, seq_len)

assert output.shape == (batch_size, seq_len, hidden_dim)
assert weights.shape == (batch_size, num_heads, seq_len, seq_len)
print("Multi-Head Attention: Shapes validated successfully!")

# Demonstrate with residual connection and dropout
class MultiHeadAttentionBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout_rate):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_dim)
        self.attn = MultiHeadAttention(hidden_dim, num_heads, dropout_rate)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x, mask=None):
        # Residual connection: x + dropout(attn_output)
        normalized_x = self.norm(x)
        attn_output, _ = self.attn(normalized_x, normalized_x, normalized_x, mask)
        # Apply dropout to the attention output before adding to residual
        output = x + self.dropout(attn_output)
        return output

print("\n--- Multi-Head Attention Block with Residual and Dropout ---")
attn_block = MultiHeadAttentionBlock(hidden_dim, num_heads, dropout_rate)
block_output = attn_block(dummy_input, mask=dummy_mask)

print(f"Input shape: {dummy_input.shape}")
print(f"Block Output shape: {block_output.shape}")
assert block_output.shape == (batch_size, seq_len, hidden_dim)
print("Multi-Head Attention Block: Shapes validated successfully with residual and dropout!")


--- Multi-Head Attention Validation ---
Input shape: torch.Size([2, 10, 256])
Output shape: torch.Size([2, 10, 256])
Attention Weights shape: torch.Size([2, 8, 10, 10])
Multi-Head Attention: Shapes validated successfully!

--- Multi-Head Attention Block with Residual and Dropout ---
Input shape: torch.Size([2, 10, 256])
Block Output shape: torch.Size([2, 10, 256])
Multi-Head Attention Block: Shapes validated successfully with residual and dropout!


### Custom Encoder Stack & Training Loop

Now, let's proceed with the optional task of building a lightweight encoder-only network and a training loop. We'll start by processing the dataset.

In [6]:
import pandas as pd

# Use the existing extraction_path from the notebook
zip_file_path_train = os.path.join(
    extraction_path,
    'Basics of BERT and XLM-RoBERTa - PyTorch',
    'train.csv.zip'
)
extraction_path_train = os.path.join(
    extraction_path,
    'Basics of BERT and XLM-RoBERTa - PyTorch'
)

with zipfile.ZipFile(zip_file_path_train, 'r') as zip_ref:
    zip_ref.extractall(extraction_path_train)

print(f"'{zip_file_path_train}' unzipped to '{extraction_path_train}'")

# Load the training data
try:
    train_df = pd.read_csv(os.path.join(extraction_path_train, 'train.csv'))
    print("Train data loaded successfully. Head of the dataframe:")
    print(train_df.head())
    print(f"Train dataframe shape: {train_df.shape}")
except FileNotFoundError:
    print(f"Error: train.csv not found at {os.path.join(extraction_path_train, 'train.csv')}")
    print(f"Contents of {extraction_path_train}:")
    for root, dirs, files in os.walk(extraction_path_train):
        for name in files:
            print(os.path.join(root, name))
        for name in dirs:
            print(os.path.join(root, name))

'/unzipped_data\Basics of BERT and XLM-RoBERTa - PyTorch\train.csv.zip' unzipped to '/unzipped_data\Basics of BERT and XLM-RoBERTa - PyTorch'
Train data loaded successfully. Head of the dataframe:
           id                                            premise  \
0  5130fd2cb5  and these comments were considered in formulat...   
1  5b72532a0b  These are issues that we wrestle with in pract...   
2  3931fbe82a  Des petites choses comme celles-là font une di...   
3  5622f0c60b  you know they can't really defend themselves l...   
4  86aaa48b45  ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...   

                                          hypothesis lang_abv language  label  
0  The rules developed in the interim were put to...       en  English      0  
1  Practice groups are not permitted to work on t...       en  English      2  
2              J'essayais d'accomplir quelque chose.       fr   French      0  
3  They can't defend themselves because of their ...       en  English    

Now, we'll define the components for our Encoder Stack: a FeedForward layer and an Encoder Block that combines MultiHeadAttention with Layer Normalization and FeedForward layers.

In [7]:
import torch
import torch.nn as nn

class FeedForward(nn.Module):
    def __init__(self, hidden_dim, ff_dim, dropout_rate=0.1):
        super().__init__()
        self.fc1 = nn.Linear(hidden_dim, ff_dim)
        self.fc2 = nn.Linear(ff_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

class EncoderBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout_rate=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(hidden_dim, num_heads, dropout_rate)
        self.feed_forward = FeedForward(hidden_dim, ff_dim, dropout_rate)

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x, mask=None):
        # Multi-Head Attention with Residual Connection and Layer Norm
        _x = self.norm1(x) # Apply LayerNorm before attention (pre-norm)
        attn_output, _ = self.self_attn(_x, _x, _x, mask)
        x = x + self.dropout(attn_output) # Add residual connection and dropout

        # Feed-Forward with Residual Connection and Layer Norm
        _x = self.norm2(x) # Apply LayerNorm before feed-forward
        ff_output = self.feed_forward(_x)
        x = x + self.dropout(ff_output) # Add residual connection and dropout

        return x

class Encoder(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_layers, num_heads, ff_dim, dropout_rate, max_seq_len):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, hidden_dim)
        self.position_embedding = nn.Embedding(max_seq_len, hidden_dim)
        self.layers = nn.ModuleList([
            EncoderBlock(hidden_dim, num_heads, ff_dim, dropout_rate)
            for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout_rate)
        self.scale = torch.sqrt(torch.tensor(hidden_dim, dtype=torch.float32))

    def forward(self, src, src_mask):
        batch_size, seq_len = src.shape

        # Create position tensor
        pos = torch.arange(0, seq_len).unsqueeze(0).repeat(batch_size, 1).to(src.device)

        # Embed tokens and positions, then sum them
        # Scale embeddings similar to original Transformer for stability
        src = self.dropout((self.token_embedding(src) * self.scale) + self.position_embedding(pos))

        for layer in self.layers:
            src = layer(src, src_mask)

        return src

print("Encoder Stack components (FeedForward, EncoderBlock, Encoder) defined.")

Encoder Stack components (FeedForward, EncoderBlock, Encoder) defined.


Next, we'll set up the data processing for the NLI dataset, which involves tokenization and creating PyTorch DataLoaders. We'll use Hugging Face's `transformers` library for tokenization to align with the BERT/XLM-RoBERTa context mentioned.

In [9]:
# Install transformers if not already installed
try:
    import transformers
except ImportError:
    !pip install transformers datasets accelerate
    import transformers

from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

# Load a pre-trained tokenizer (e.g., 'bert-base-uncased')
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Define a custom Dataset class for NLI
class NLIDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.data = dataframe
        self.premises = dataframe['premise']
        self.hypotheses = dataframe['hypothesis']
        self.labels = dataframe['label']
        self.max_len = max_len

    def __len__(self):
        return len(self.premises)

    def __getitem__(self, index):
        premise = str(self.premises[index])
        hypothesis = str(self.hypotheses[index])
        label = self.labels[index]

        # Tokenize premise and hypothesis
        inputs = self.tokenizer(premise, hypothesis,
                                add_special_tokens=True,
                                max_length=self.max_len,
                                padding='max_length',
                                truncation=True,
                                return_tensors='pt')

        input_ids = inputs['input_ids'].squeeze(0) # Remove batch dimension added by return_tensors='pt'
        attention_mask = inputs['attention_mask'].squeeze(0)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Prepare the dataset
# Assuming 'train_df' is already loaded and has 'premise', 'hypothesis', 'label' columns
# Filter out rows where 'label' is -1 if present (often indicates neutral or unk in NLI)
if 'label' in train_df.columns:
    train_df = train_df[train_df['label'] != -1].reset_index(drop=True)
    print(f"Filtered train_df shape (excluding label -1): {train_df.shape}")

# Split data into training and validation (simple split for demonstration)
train_size = int(0.8 * len(train_df))
val_size = len(train_df) - train_size

train_data = train_df.iloc[:train_size].reset_index(drop=True)
val_data = train_df.iloc[train_size:].reset_index(drop=True)

MAX_LEN = 128 # Maximum sequence length for tokenization
BATCH_SIZE = 32

train_dataset = NLIDataset(train_data, tokenizer, MAX_LEN)
val_dataset = NLIDataset(val_data, tokenizer, MAX_LEN)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print("DataLoaders created.")

# Example of one batch
for batch in train_dataloader:
    print("\nExample batch from DataLoader:")
    print(f"Input IDs shape: {batch['input_ids'].shape}")
    print(f"Attention Mask shape: {batch['attention_mask'].shape}")
    print(f"Labels shape: {batch['labels'].shape}")
    break

c:\Users\meles\Documents\TTA_DI_BootCamp_Gilles-Chris_MAKE\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\meles\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Filtered train_df shape (excluding label -1): (12120, 6)
Train dataset size: 9696
Validation dataset size: 2424
DataLoaders created.

Example batch from DataLoader:
Input IDs shape: torch.Size([32, 128])
Attention Mask shape: torch.Size([32, 128])
Labels shape: torch.Size([32])


Finally, we will set up the training loop for our custom encoder-only network on the NLI dataset.

In [11]:
import torch.optim as optim
from tqdm.notebook import tqdm
from tqdm.auto import tqdm

# Hyperparameters for the Encoder and Training
VOCAB_SIZE = tokenizer.vocab_size
HIDDEN_DIM = 256 # Must be divisible by num_heads
NUM_LAYERS = 2
NUM_HEADS = 8
FF_DIM = HIDDEN_DIM * 4 # Typically 4 times hidden_dim
DROPOUT_RATE = 0.1
MAX_SEQ_LEN = MAX_LEN # From tokenizer setup
NUM_CLASSES = 3 # NLI has 3 classes: entailment, neutral, contradiction

# Instantiate the Encoder model
encoder = Encoder(VOCAB_SIZE, HIDDEN_DIM, NUM_LAYERS, NUM_HEADS, FF_DIM, DROPOUT_RATE, MAX_SEQ_LEN)

# For NLI classification, we need a classification head on top of the encoder output
class NLIClassifier(nn.Module):
    def __init__(self, encoder, hidden_dim, num_classes):
        super().__init__()
        self.encoder = encoder
        # Global average pooling or simply take the [CLS] token output
        # We'll take the [CLS] token output (first token) for simplicity, common in BERT-like models
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids, attention_mask):
        # The attention mask needs to be adapted for the MultiHeadAttention in our EncoderBlock
        # Our MHA expects (batch_size, 1, 1, seq_len) or (batch_size, 1, query_len, key_len)
        # Transformer attention mask is typically (batch_size, seq_len) or (batch_size, 1, seq_len)
        # We need to expand it to (batch_size, 1, 1, seq_len) or (batch_size, 1, seq_len, seq_len) for self-attention
        # For now, let's assume `attention_mask` is (batch_size, seq_len) from tokenizer.
        # We'll create a causal mask (upper triangular) for decoder, but for encoder just block padding tokens.
        # Our MultiHeadAttention expects a mask of shape (batch_size, query_len, key_len) or broadcastable

        # Create an appropriate mask for the encoder:
        # Original attention_mask from tokenizer is (batch_size, seq_len) where 0 means masked
        # We need to expand it to (batch_size, 1, 1, seq_len) for broadcasting across heads and query positions
        # It should be 1 where tokens are present, 0 where padded
        mask_for_encoder = attention_mask.unsqueeze(1).unsqueeze(2) # Shape: (batch_size, 1, 1, seq_len)

        encoder_output = self.encoder(input_ids, mask_for_encoder)

        # Take the output corresponding to the [CLS] token (first token)
        cls_output = encoder_output[:, 0, :]
        logits = self.fc(cls_output)
        return logits

model = NLIClassifier(encoder, HIDDEN_DIM, NUM_CLASSES)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

N_EPOCHS = 3 # Train for a few epochs

print("\nStarting Training Loop...")

for epoch in range(N_EPOCHS):
    model.train()
    train_loss = 0.0
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1} Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_dataloader)

    # Validation phase
    model.eval()
    val_loss = 0.0
    correct_predictions = 0
    total_predictions = 0

    with torch.no_grad():
        for batch in tqdm(val_dataloader, desc=f"Epoch {epoch+1} Validation"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total_predictions += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    val_loss /= len(val_dataloader)
    val_accuracy = correct_predictions / total_predictions

    print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}, Val Accuracy = {val_accuracy:.4f}")

print("Training complete.")


Starting Training Loop...


Epoch 1 Validation: 100%|██████████| 76/76 [00:09<00:00,  8.14it/s]


Epoch 1: Train Loss = 2.8215, Val Loss = 1.4883, Val Accuracy = 0.3111


Epoch 2 Validation: 100%|██████████| 76/76 [00:17<00:00,  4.39it/s]


Epoch 2: Train Loss = 2.0807, Val Loss = 1.3264, Val Accuracy = 0.3106


Epoch 3 Validation: 100%|██████████| 76/76 [00:09<00:00,  8.08it/s]

Epoch 3: Train Loss = 1.5774, Val Loss = 1.1084, Val Accuracy = 0.3606
Training complete.
